## **Delta Lake (Ideal for transactional databases, advanced analytics, and version control)**
Delta Lake adds ACID transactions on top of Parquet files.

In [1]:
# Install compatible versions of PySpark and Delta Lake to avoid conflicts
!pip install -q pyspark==3.5.4 delta-spark==3.2.0

In [3]:
# Mount your Google Drive
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Initialize PySpark with Delta Lake support

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Configure the builder with the extensions and the Delta catalog
builder = SparkSession.builder \
    .appName("Colab_Session_10") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Initialize the session by automatically downloading the correct Delta Lake package
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# Define the schema
schema = StructType() \
    .add("ProductID", IntegerType(), True) \
    .add("ProductName", StringType(), True) \
    .add("Category", StringType(), True) \
    .add("ListPrice", DoubleType(), True)


In [6]:
# Display the DF
df = spark.read.format("csv").option("header","True").load("/content/drive/MyDrive/Mis_CSV/products.csv")
df.show()    # In Fabric use display(df)

+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      772|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      773|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      774|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      775|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      776|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      777|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      778|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      779|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      780|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      781|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      782|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      783|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      784|Mountain-200 Blac...|Mountain Bikes|2294.9900|
|      785|Mou

In [7]:
# 1. Leer los datos aplicando el esquema definido previamente para asegurar los tipos de datos correctos
df_products = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/content/drive/MyDrive/Mis_CSV/products.csv")

# 2. Guardar el DataFrame directamente como Tabla Delta en Google Drive
df_products.write.format("delta").mode("overwrite").save(
    "/content/drive/MyDrive/mis_tablas_deltalake/products_delta"
)

print("¡La tabla Delta se ha guardado con éxito en tu Google Drive usando la sesión con soporte de Delta activo!")

¡La tabla Delta se ha guardado con éxito en tu Google Drive usando la sesión con soporte de Delta activo!


In [8]:
# 1. Read the Delta table saved in Google Drive
ruta_delta = "/content/drive/MyDrive/mis_tablas_deltalake/products_delta"
df_delta_leido = spark.read.format("delta").load(ruta_delta)

# 2. Display the content and its schema for verification
df_delta_leido.show(10, truncate=False)
df_delta_leido.printSchema()

+---------+-----------------------+--------------+---------+
|ProductID|ProductName            |Category      |ListPrice|
+---------+-----------------------+--------------+---------+
|771      |Mountain-100 Silver, 38|Mountain Bikes|3399.99  |
|772      |Mountain-100 Silver, 42|Mountain Bikes|3399.99  |
|773      |Mountain-100 Silver, 44|Mountain Bikes|3399.99  |
|774      |Mountain-100 Silver, 48|Mountain Bikes|3399.99  |
|775      |Mountain-100 Black, 38 |Mountain Bikes|3374.99  |
|776      |Mountain-100 Black, 42 |Mountain Bikes|3374.99  |
|777      |Mountain-100 Black, 44 |Mountain Bikes|3374.99  |
|778      |Mountain-100 Black, 48 |Mountain Bikes|3374.99  |
|779      |Mountain-200 Silver, 38|Mountain Bikes|2319.99  |
|780      |Mountain-200 Silver, 42|Mountain Bikes|2319.99  |
+---------+-----------------------+--------------+---------+
only showing top 10 rows

root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullabl